In [ ]:
"""
  SMART ACADEMIC MENTOR
  AI & Machine Learning Based Student Performance Analysis System
  AI/ML Project

  Dataset: 300 students · 14 features
  Best Model: SVM (RBF) — Test 83.33% | CV 82.99% ± 1.32%
"""


## Smart Academic Mentor — Updated Pipeline

| | Original | Updated |
|---|---|---|
| **Dataset size** | 50 students | **300 students** |
| **Train / Test** | 35 / 15 | **210 / 90** |
| **Best model** | SVM 86.67% (CV ±10.69%) | **SVM 83.33% (CV 82.99% ±1.32%)** |
| **Overfitting** | Large CV–test gap | **Near-zero gap** |

> **Run all cells** to regenerate all visualisations with the 300-student `student_data.csv`.


In [ ]:

# IMPORTS 

import warnings
warnings.filterwarnings("ignore")

# Compatible with both Jupyter and VS Code notebooks
import matplotlib
matplotlib.use("module://matplotlib_inline.backend_inline")  # works in both
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns

from matplotlib.patches import FancyBboxPatch
from matplotlib.lines import Line2D

import pandas as pd
import numpy as np

from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.cluster import KMeans
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.metrics import (
    accuracy_score, confusion_matrix,
    classification_report, ConfusionMatrixDisplay
)
from sklearn.decomposition import PCA

import os, sys
import glob


In [ ]:
#  GLOBAL STYLE CONFIG
plt.rcParams.update({
    "figure.dpi"        : 150,
    "figure.facecolor"  : "#0D1117",
    "axes.facecolor"    : "#161B22",
    "axes.edgecolor"    : "#30363D",
    "axes.labelcolor"   : "#C9D1D9",
    "axes.titlecolor"   : "#F0F6FC",
    "axes.titlesize"    : 13,
    "axes.labelsize"    : 10,
    "xtick.color"       : "#8B949E",
    "ytick.color"       : "#8B949E",
    "text.color"        : "#C9D1D9",
    "legend.facecolor"  : "#21262D",
    "legend.edgecolor"  : "#30363D",
    "legend.labelcolor" : "#C9D1D9",
    "grid.color"        : "#21262D",
    "grid.linewidth"    : 0.6,
    "font.family"       : "DejaVu Sans",
    "savefig.facecolor" : "#0D1117",
    "savefig.bbox"      : "tight",
    "savefig.pad_inches": 0.3,
})

PRIORITY_RANK = {"Low": 0, "Medium": 1, "High": 2, "Critical": 3, "Excellent": -1}

def _higher_priority(a: str, b: str) -> str:
    """Return whichever priority string ranks higher."""
    return a if PRIORITY_RANK.get(a, 0) >= PRIORITY_RANK.get(b, 0) else b

PALETTE   = ["#58A6FF", "#3FB950", "#F78166", "#D2A8FF", "#FFA657"]
OUT_DIR   = "outputs"
os.makedirs(OUT_DIR, exist_ok=True)

SUBJECTS = ["Maths", "Physics", "Chemistry", "Programming", "English"]
WEAK_THRESHOLD = 55  # marks below this are flagged as weak

In [ ]:
#  SECTION 1 ▶ DATA LOADING
def load_data(path: str) -> pd.DataFrame:
    """Load and display a preview of the raw dataset."""
    print("\n" + "═"*62)
    print("  MODULE 1 ▷  DATA LOADING")
    print("═"*62)
    df = pd.read_csv(path)
    print(f"  ✔  Dataset loaded  →  {df.shape[0]} students, {df.shape[1]} features")
    print(f"\n  COLUMNS : {list(df.columns)}")
    print(f"\n  DATA PREVIEW (first 5 rows):\n")
    print(df.head().to_string(index=False))
    print(f"\n  BASIC STATISTICS:\n")
    print(df.describe(include="all").to_string())
    return df

In [ ]:
#  SECTION 2 ▶ DATA PREPROCESSING
def preprocess_data(df: pd.DataFrame):
    print("\n" + "═"*62)
    print("  MODULE 2 ▷  DATA PREPROCESSING")
    print("═"*62)

    df = df.copy()

    # Step 1: Drop identifiers, subject scores, and leaky features 
    drop_cols = ["StudentID", "Name", "Internal_Marks"] + SUBJECTS
    df.drop(columns=drop_cols, inplace=True)
    print("  ✔  Dropped: StudentID, Name")
    print("  ✔  Dropped subject scores (data leakage)")
    print("  ✔  Dropped Internal_Marks (direct leakage – determines pass/fail)")

    # Step 2: Missing value handling 
    missing_before = df.isnull().sum().sum()
    for col in df.select_dtypes(include=[np.number]).columns:
        df[col].fillna(df[col].mean(), inplace=True)
    print(f"  ✔  Missing values found: {missing_before}  →  Filled with column mean")

    # Step 3: Encode categoricals 
    le_extra = LabelEncoder()
    df["Extracurricular"] = le_extra.fit_transform(df["Extracurricular"])
    print(f"  ✔  Encoded 'Extracurricular': No=0, Yes=1")

    le_result = LabelEncoder()
    df["Final_Result"] = le_result.fit_transform(df["Final_Result"])
    print(f"  ✔  Encoded 'Final_Result': Fail=0, Pass=1")
    print(f"     Label mapping → {dict(zip(le_result.classes_, le_result.transform(le_result.classes_)))}")

    # Step 4: Feature / Target split 
    y = df["Final_Result"]
    X = df.drop(columns=["Final_Result"])

    # Drop highly correlated features (threshold 0.90, tightened from 0.95)
    corr = X.corr().abs()
    upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
    to_drop = [col for col in upper.columns if any(upper[col] > 0.90)]
    X = X.drop(columns=to_drop)
    print(f"  ✔  Dropped highly correlated features: {to_drop if to_drop else 'None'}")

    feature_names = list(X.columns)
    print(f"  ✔  Features used for classification: {feature_names}")
    print(f"  ✔  Final feature matrix shape: {X.shape}")
    print(f"  ✔  Class distribution → Pass: {(y==1).sum()}  |  Fail: {(y==0).sum()}")

    # NOTE: Scaling is done inside train_and_compare_models after the split
    #       to avoid leakage. No random noise added – it masked real signals.
    return X.values, y.values, X, None, le_result, feature_names

In [ ]:
#  SECTION 3 ▶ K-MEANS CLUSTERING
def kmeans_clustering(X_scaled: np.ndarray, df_orig: pd.DataFrame, feature_names: list):
    """
    Unsupervised learning with K-Means (k=3):
      Cluster 0 → At-Risk Students
      Cluster 1 → Average Performers
      Cluster 2 → High Performers
    Visualises clusters with PCA reduction to 2D.
    """
    print("\n" + "═"*62)
    print("  MODULE 3 ▷  K-MEANS CLUSTERING  (Unsupervised Learning)")
    print("═"*62)

    # Elbow method 
    inertias = []
    K_range  = range(1, 10)
    for k in K_range:
        km = KMeans(n_clusters=k, random_state=42, n_init=10)
        km.fit(X_scaled)
        inertias.append(km.inertia_)

    # Fit final model 
    kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
    labels = kmeans.fit_predict(X_scaled)

    # Map cluster to category by mean attendance
    df_temp = df_orig.copy()
    df_temp["Cluster"] = labels
    cluster_means = df_temp.groupby("Cluster")["Attendance"].mean()
    sorted_clusters = cluster_means.sort_values().index.tolist()
    cluster_map = {
        sorted_clusters[0]: "At-Risk",
        sorted_clusters[1]: "Average",
        sorted_clusters[2]: "High Performer"
    }
    df_temp["Category"] = df_temp["Cluster"].map(cluster_map)

    counts = df_temp["Category"].value_counts()
    for cat, cnt in counts.items():
        print(f"  ✔  {cat:18s} →  {cnt:3d} students")

    # PCA for 2D visualisation 
    pca = PCA(n_components=2)
    X_2d = pca.fit_transform(X_scaled)
    print(f"\n  PCA variance explained: {pca.explained_variance_ratio_.sum()*100:.1f}%")

    # PLOT 1: Elbow Curve + Cluster Scatter 
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    fig.suptitle("K-Means Clustering Analysis", fontsize=16, color="#F0F6FC",
                 fontweight="bold", y=1.02)

    # Elbow
    ax = axes[0]
    ax.plot(K_range, inertias, "o-", color="#58A6FF", linewidth=2.5,
            markersize=8, markerfacecolor="#F78166", markeredgecolor="#F0F6FC",
            markeredgewidth=1.5)
    ax.axvline(3, color="#3FB950", linestyle="--", linewidth=1.5,
               label="Optimal k = 3")
    ax.set_title("Elbow Method – Optimal k", pad=14)
    ax.set_xlabel("Number of Clusters (k)")
    ax.set_ylabel("Inertia (WCSS)")
    ax.legend()
    ax.grid(True, alpha=0.4)

    # Scatter
    ax2 = axes[1]
    colors_map = {"At-Risk": "#F78166", "Average": "#FFA657",
                  "High Performer": "#3FB950"}
    for cat, grp in df_temp.groupby("Category"):
        mask = df_temp["Category"] == cat
        ax2.scatter(X_2d[mask, 0], X_2d[mask, 1],
                    label=cat, color=colors_map[cat],
                    s=90, alpha=0.85, edgecolors="#0D1117", linewidths=0.5)
    ax2.set_title("Student Clusters (PCA 2D Projection)", pad=14)
    ax2.set_xlabel(f"PCA Component 1 ({pca.explained_variance_ratio_[0]*100:.1f}% var)")
    ax2.set_ylabel(f"PCA Component 2 ({pca.explained_variance_ratio_[1]*100:.1f}% var)")
    ax2.legend(loc="upper right")
    ax2.grid(True, alpha=0.4)

    plt.tight_layout()
    plt.savefig(f"{OUT_DIR}/01_kmeans_clustering.png")
    plt.show()
    # plt.close()
    print(f"\n  ✔  Cluster plot saved → {OUT_DIR}/01_kmeans_clustering.png")

    # PLOT 2: Cluster Profile Radar / Bar 
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    fig.suptitle("Cluster Profiles – Feature Averages", fontsize=15,
                 color="#F0F6FC", fontweight="bold")

    numeric_features = ["Attendance", "Internal_Marks", "Assignment_Submission",
                        "Previous_CGPA", "Study_Hours"]
    cluster_profile = df_temp.groupby("Category")[numeric_features].mean()

    x_pos = np.arange(len(numeric_features))
    width = 0.25
    cat_order = ["At-Risk", "Average", "High Performer"]
    bar_colors = [colors_map[c] for c in cat_order]

    ax = axes[0]
    for i, (cat, col) in enumerate(zip(cat_order, bar_colors)):
        if cat in cluster_profile.index:
            ax.bar(x_pos + i*width, cluster_profile.loc[cat, numeric_features],
                   width, label=cat, color=col, alpha=0.85, edgecolor="#0D1117")
    ax.set_xticks(x_pos + width)
    ax.set_xticklabels(numeric_features, rotation=20, ha="right", fontsize=9)
    ax.set_title("Feature Averages by Cluster Category")
    ax.set_ylabel("Average Value")
    ax.legend()
    ax.grid(axis="y", alpha=0.4)

    ax2 = axes[1]
    counts_ordered = [counts.get(c, 0) for c in cat_order]
    wedge_colors   = [colors_map[c] for c in cat_order]
    wedges, texts, autotexts = ax2.pie(
        counts_ordered, labels=cat_order, colors=wedge_colors,
        autopct="%1.1f%%", startangle=140,
        wedgeprops={"edgecolor": "#0D1117", "linewidth": 1.5},
        textprops={"color": "#C9D1D9", "fontsize": 10}
    )
    for at in autotexts:
        at.set_color("#0D1117")
        at.set_fontweight("bold")
    ax2.set_title("Distribution of Student Categories")

    plt.tight_layout()
    plt.savefig(f"{OUT_DIR}/02_cluster_profiles.png")
    plt.show()
    # plt.close()
    print(f"  ✔  Cluster profile plot saved → {OUT_DIR}/02_cluster_profiles.png")

    return labels, cluster_map, df_temp


In [ ]:
#  SECTION 4 ▶ SUPERVISED LEARNING – TRAIN & COMPARE 4 CLASSIFIERS
def train_and_compare_models(X: np.ndarray, y: np.ndarray):

    print("\n" + "═"*62)
    print("  MODULE 4 ▷  SUPERVISED LEARNING – MODEL COMPARISON")
    print("═"*62)

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.30, random_state=42, stratify=y
    )

    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train)
    X_test  = scaler.transform(X_test)

    print(f"  Train samples: {len(X_train)}  |  Test samples: {len(X_test)}")

    models = {
        "Logistic Regression": LogisticRegression(
            max_iter=1000, C=0.1, class_weight="balanced", random_state=42
        ),
        "Decision Tree"      : DecisionTreeClassifier(
            max_depth=2, min_samples_leaf=4, class_weight="balanced", random_state=42
        ),
        "Random Forest"      : RandomForestClassifier(
            n_estimators=50, max_depth=3, min_samples_leaf=3,
            class_weight="balanced", random_state=42
        ),
        "SVM"                : SVC(
            kernel="rbf", C=0.5, gamma="scale",
            probability=True, class_weight="balanced", random_state=42
        ),
    }

    results = {}

    print(f"\n  {'Model':<22} {'Test Acc':>10}  {'CV Mean':>10}  {'CV Std':>8}")
    print(f"  {'-'*56}")

    from sklearn.model_selection import StratifiedKFold
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

    for name, model in models.items():
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)
        acc    = accuracy_score(y_test, y_pred)
        cv     = cross_val_score(model, X_train, y_train, cv=skf, scoring="accuracy")
        cm     = confusion_matrix(y_test, y_pred)
        cr     = classification_report(y_test, y_pred, target_names=["Fail","Pass"], output_dict=True)

        results[name] = {
            "model"   : model,
            "accuracy": acc,
            "cv_mean" : cv.mean(),
            "cv_std"  : cv.std(),
            "y_pred"  : y_pred,
            "cm"      : cm,
            "report"  : cr,
        }

        print(f"\n  {name}")
        print(classification_report(y_test, y_pred, target_names=["Fail","Pass"]))
        print(f"  {name:<22} {acc*100:>9.2f}%  {cv.mean()*100:>9.2f}%  {cv.std()*100:>7.2f}%")

    best = max(results, key=lambda k: results[k]["cv_mean"])
    print(f"\n  ✔  Best model (by CV mean): {best}")

    # ── PLOT A: Classification Metrics Heatmap ────────────────────────────────
    model_names = list(results.keys())
    metrics_data = np.array([
        [
            results[m]["report"]["Fail"]["precision"],
            results[m]["report"]["Fail"]["recall"],
            results[m]["report"]["Fail"]["f1-score"],
            results[m]["accuracy"],
        ]
        for m in model_names
    ])

    fig, ax = plt.subplots(figsize=(10, 5))
    fig.suptitle("Classification Metrics Heatmap", fontsize=15,
                 color="#F0F6FC", fontweight="bold")
    im = ax.imshow(metrics_data, cmap="YlGnBu", vmin=0, vmax=1, aspect="auto")
    ax.set_xticks([0, 1, 2, 3])
    ax.set_xticklabels(["Precision", "Recall", "F1-Score", "Accuracy"],
                       fontsize=11, color="#C9D1D9")
    ax.set_yticks(range(len(model_names)))
    ax.set_yticklabels(model_names, fontsize=11, color="#C9D1D9")
    ax.set_ylabel("Model", fontsize=11, color="#8B949E")
    cbar = plt.colorbar(im, ax=ax, shrink=0.85)
    cbar.ax.tick_params(colors="#C9D1D9")
    for i in range(len(model_names)):
        for j in range(4):
            val = metrics_data[i, j]
            ax.text(j, i, f"{val:.3f}", ha="center", va="center",
                    fontsize=12, fontweight="bold",
                    color="white" if val < 0.65 else "#0D1117")
    plt.tight_layout()
    plt.savefig(f"{OUT_DIR}/04a_classification_metrics_heatmap.png", dpi=150)
    plt.show()
    print(f"  ✔  Classification metrics heatmap saved → {OUT_DIR}/04a_classification_metrics_heatmap.png")

    # ── PLOT B: Model Performance Comparison (Test Acc + CV Acc side by side) ─
    test_accs = [results[m]["accuracy"] * 100 for m in model_names]
    cv_means  = [results[m]["cv_mean"]  * 100 for m in model_names]
    cv_stds   = [results[m]["cv_std"]   * 100 for m in model_names]

    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    fig.suptitle("Model Performance Comparison", fontsize=15,
                 color="#F0F6FC", fontweight="bold")

    bar_colors = ["#58A6FF", "#3FB950", "#F78166", "#D2A8FF"]
    x = np.arange(len(model_names))

    # Left: Test Accuracy
    ax1 = axes[0]
    ax1.set_title("Test Accuracy (%)", fontsize=12, color="#F0F6FC", pad=10)
    bars1 = ax1.bar(x, test_accs, color=bar_colors, edgecolor="#0D1117",
                    linewidth=1, alpha=0.9)
    for bar, val in zip(bars1, test_accs):
        ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                 f"{val:.1f}%", ha="center", fontsize=10,
                 color="#F0F6FC", fontweight="bold")
    ax1.set_xticks(x)
    ax1.set_xticklabels([m.replace(" ", "\n") for m in model_names], fontsize=9)
    ax1.set_ylim(0, 130)
    ax1.set_ylabel("Accuracy (%)", fontsize=10)
    ax1.grid(axis="y", alpha=0.4)
    ax1.axhline(100, color="#FFA657", linestyle="--", linewidth=1, alpha=0.5)

    # Right: 5-Fold CV Accuracy with error bars
    ax2 = axes[1]
    ax2.set_title("5-Fold Cross-Validation Accuracy (%)", fontsize=12,
                  color="#F0F6FC", pad=10)
    bars2 = ax2.bar(x, cv_means, color=bar_colors, edgecolor="#0D1117",
                    linewidth=1, alpha=0.9,
                    yerr=cv_stds, capsize=6,
                    error_kw={"ecolor": "#FFA657", "elinewidth": 2, "capthick": 2})
    for bar, val in zip(bars2, cv_means):
        ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1.5,
                 f"{val:.1f}%", ha="center", fontsize=10,
                 color="#F0F6FC", fontweight="bold")
    ax2.set_xticks(x)
    ax2.set_xticklabels([m.replace(" ", "\n") for m in model_names], fontsize=9)
    ax2.set_ylim(0, 130)
    ax2.set_ylabel("CV Accuracy (%)", fontsize=10)
    ax2.grid(axis="y", alpha=0.4)
    ax2.axhline(100, color="#FFA657", linestyle="--", linewidth=1, alpha=0.5)

    # Highlight best model in both charts
    best_idx = model_names.index(best)
    for ax_tmp, bars_tmp in [(ax1, bars1), (ax2, bars2)]:
        bars_tmp[best_idx].set_edgecolor("#F0F6FC")
        bars_tmp[best_idx].set_linewidth(2.5)

    plt.tight_layout()
    plt.savefig(f"{OUT_DIR}/04b_model_performance_comparison.png", dpi=150)
    plt.show()
    print(f"  ✔  Model performance comparison saved → {OUT_DIR}/04b_model_performance_comparison.png")

    # ── PLOT C: Confusion Matrices (2×2 grid) ─────────────────────────────────
    fig, axes = plt.subplots(2, 2, figsize=(12, 10))
    fig.suptitle("Confusion Matrices – All Classifiers", fontsize=16,
                 color="#F0F6FC", fontweight="bold")
    axes = axes.flatten()

    for ax, (name, res) in zip(axes, results.items()):
        cm_val = res["cm"]
        ax.imshow(cm_val, interpolation="nearest",
                  cmap=sns.color_palette("Blues", as_cmap=True))
        title_str = "{}\nAcc: {:.1f}%  |  CV: {:.1f}% +/- {:.1f}%".format(
            name, res["accuracy"]*100, res["cv_mean"]*100, res["cv_std"]*100)
        ax.set_title(title_str, fontsize=10, color="#F0F6FC", pad=8)

        ax.set_xticks([0, 1])
        ax.set_yticks([0, 1])
        ax.set_xticklabels(["Fail", "Pass"], fontsize=11, color="#C9D1D9")
        ax.set_yticklabels(["Fail", "Pass"], fontsize=11, color="#C9D1D9")
        ax.set_xlabel("Predicted Label", fontsize=10, color="#8B949E")
        ax.set_ylabel("True Label", fontsize=10, color="#8B949E")
        thresh = cm_val.max() / 2.0
        labels_cm = [["TN", "FP"], ["FN", "TP"]]
        for i in range(2):
            for j in range(2):
                ax.text(j, i - 0.15, str(cm_val[i, j]),
                        ha="center", va="center", fontsize=22, fontweight="bold",
                        color="white" if cm_val[i, j] > thresh else "#0D1117")
                ax.text(j, i + 0.22, f"({labels_cm[i][j]})",
                        ha="center", va="center", fontsize=9,
                        color="white" if cm_val[i, j] > thresh else "#0D1117")
        if name == best:
            for spine in ax.spines.values():
                spine.set_edgecolor("#3FB950")
                spine.set_linewidth(2.5)

    plt.tight_layout(rect=[0, 0, 1, 0.96])
    plt.savefig(f"{OUT_DIR}/04c_confusion_matrices.png", dpi=150)
    plt.show()
    print(f"  ✔  Confusion matrices saved → {OUT_DIR}/04c_confusion_matrices.png")

    return results, X_train, X_test, y_train, y_test


In [ ]:
#  SECTION 5 ▶ FEATURE IMPORTANCE (Random Forest)
def plot_feature_importance(results: dict, feature_names: list):
    """Extract and plot feature importances from the Random Forest model."""
    print("\n" + "═"*62)
    print("  MODULE 5 ▷  FEATURE IMPORTANCE ANALYSIS")
    print("═"*62)

    rf_model      = results["Random Forest"]["model"]
    importances   = rf_model.feature_importances_
    indices       = np.argsort(importances)[::-1]
    sorted_names  = [feature_names[i] for i in indices]
    sorted_imp    = importances[indices]

    print(f"\n  Feature Rankings:")
    for rank, (name, imp) in enumerate(zip(sorted_names, sorted_imp), 1):
        bar = "█" * int(imp * 80)
        print(f"  {rank:2}. {name:<26} {imp:.4f}  {bar}")

    fig, ax = plt.subplots(figsize=(10, 6))
    bars = ax.barh(sorted_names[::-1], sorted_imp[::-1],
                   color=sns.color_palette("Blues_r", len(sorted_names)),
                   edgecolor="#0D1117", linewidth=0.8)
    for bar, imp in zip(bars, sorted_imp[::-1]):
        ax.text(bar.get_width() + 0.002, bar.get_y() + bar.get_height()/2,
                f"{imp:.4f}", va="center", fontsize=9, color="#C9D1D9")
    ax.set_title("Feature Importance – Random Forest Classifier", pad=14,
                 fontsize=14, color="#F0F6FC")
    ax.set_xlabel("Importance Score")
    ax.grid(axis="x", alpha=0.4)
    ax.set_xlim(0, sorted_imp.max() + 0.06)
    plt.tight_layout()
    plt.savefig(f"{OUT_DIR}/06_feature_importance.png")
    plt.show()
    # plt.close()
    print(f"\n  ✔  Feature importance plot saved → {OUT_DIR}/06_feature_importance.png")


In [ ]:
#  SECTION 6 ▶ WEAK SUBJECT IDENTIFICATION
def identify_weak_subjects(df: pd.DataFrame):
    """
    Computes class-wide average per subject and flags subjects
    below the defined WEAK_THRESHOLD. Also plots subject performance.
    """
    print("\n" + "═"*62)
    print("  MODULE 6 ▷  WEAK SUBJECT IDENTIFICATION")
    print("═"*62)

    subject_means = df[SUBJECTS].mean().sort_values()
    print(f"\n  Subject Average Scores (out of 100):")
    for subj, mean in subject_means.items():
        flag = " ◄ WEAK" if mean < WEAK_THRESHOLD else ""
        bar  = "█" * int(mean / 3)
        print(f"  {subj:<14} {mean:5.1f}  {bar}{flag}")

    weak_subjects = subject_means[subject_means < WEAK_THRESHOLD].index.tolist()
    print(f"\n  ⚠  Weak Subjects Identified: {weak_subjects if weak_subjects else 'None – All above threshold'}")

    # ── PLOT 6: Subject distribution boxplots ────────────────────────────────
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    fig.suptitle("Subject-wise Performance Analysis", fontsize=15,
                 color="#F0F6FC", fontweight="bold")

    ax = axes[0]
    subject_data = [df[s].values for s in SUBJECTS]
    bp = ax.boxplot(subject_data, patch_artist=True,
                    boxprops=dict(linewidth=1.5),
                    medianprops=dict(color="#F78166", linewidth=2.5),
                    whiskerprops=dict(color="#8B949E"),
                    capprops=dict(color="#8B949E"),
                    flierprops=dict(marker="o", markersize=5,
                                   markerfacecolor="#D2A8FF"))
    for patch, col in zip(bp["boxes"], PALETTE):
        patch.set_facecolor(col)
        patch.set_alpha(0.7)
    ax.axhline(WEAK_THRESHOLD, color="#F78166", linestyle="--",
               linewidth=1.5, label=f"Weak Threshold ({WEAK_THRESHOLD})")
    ax.set_xticklabels(SUBJECTS, rotation=10)
    ax.set_title("Score Distribution per Subject")
    ax.set_ylabel("Score (out of 100)")
    ax.legend()
    ax.grid(axis="y", alpha=0.4)

    ax2 = axes[1]
    bar_colors = ["#F78166" if mean < WEAK_THRESHOLD else "#3FB950"
                  for mean in subject_means.values]
    bars = ax2.bar(subject_means.index, subject_means.values,
                   color=bar_colors, alpha=0.85,
                   edgecolor="#0D1117", linewidth=1)
    ax2.axhline(WEAK_THRESHOLD, color="#FFA657", linestyle="--",
                linewidth=2, label=f"Threshold ({WEAK_THRESHOLD})")
    for bar, val in zip(bars, subject_means.values):
        ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                 f"{val:.1f}", ha="center", fontsize=10, color="#F0F6FC",
                 fontweight="bold")
    ax2.set_title("Average Score per Subject")
    ax2.set_ylabel("Average Score")
    ax2.set_ylim(0, 115)
    ax2.legend()
    ax2.grid(axis="y", alpha=0.4)

    plt.tight_layout()
    plt.savefig(f"{OUT_DIR}/07_subject_analysis.png")
    plt.show()
    # plt.close()
    print(f"  ✔  Subject analysis plot saved → {OUT_DIR}/07_subject_analysis.png")

    return weak_subjects, subject_means


In [ ]:
#  SECTION 7 ▶ PERFORMANCE DISTRIBUTION EDA
def plot_eda(df: pd.DataFrame):
    """Exploratory Data Analysis – distributions, correlations, pass/fail."""
    print("\n" + "═"*62)
    print("  MODULE 7 ▷  EXPLORATORY DATA ANALYSIS (EDA)")
    print("═"*62)

    # ── PLOT 7: EDA Overview ─────────────────────────────────────────────────
    fig = plt.figure(figsize=(16, 12))
    gs  = gridspec.GridSpec(2, 3, figure=fig, hspace=0.45, wspace=0.35)
    fig.suptitle("Exploratory Data Analysis – Smart Academic Mentor",
                 fontsize=16, color="#F0F6FC", fontweight="bold", y=1.01)

    # 1. Pass/Fail Distribution
    ax1 = fig.add_subplot(gs[0, 0])
    result_counts = df["Final_Result"].value_counts()
    wedge_colors  = ["#3FB950", "#F78166"]
    wedges, texts, autos = ax1.pie(
        result_counts.values, labels=result_counts.index,
        autopct="%1.1f%%", colors=wedge_colors, startangle=90,
        wedgeprops={"edgecolor": "#0D1117", "linewidth": 2},
        textprops={"color": "#C9D1D9"}
    )
    for a in autos:
        a.set_color("#0D1117")
        a.set_fontweight("bold")
    ax1.set_title("Pass / Fail Distribution")

    # 2. Attendance Histogram
    ax2 = fig.add_subplot(gs[0, 1])
    ax2.hist(df["Attendance"], bins=15, color="#58A6FF", alpha=0.8,
             edgecolor="#0D1117", linewidth=0.8)
    ax2.axvline(df["Attendance"].mean(), color="#FFA657", linestyle="--",
                linewidth=2, label=f"Mean: {df['Attendance'].mean():.1f}%")
    ax2.set_title("Attendance Distribution")
    ax2.set_xlabel("Attendance (%)")
    ax2.set_ylabel("Count")
    ax2.legend()
    ax2.grid(alpha=0.4)

    # 3. Study Hours vs Attendance Scatter
    ax3 = fig.add_subplot(gs[0, 2])
    pass_mask = df["Final_Result"] == "Pass"
    ax3.scatter(df.loc[pass_mask, "Study_Hours"],
                df.loc[pass_mask, "Attendance"],
                color="#3FB950", label="Pass", alpha=0.75,
                s=60, edgecolors="#0D1117", linewidths=0.5)
    ax3.scatter(df.loc[~pass_mask, "Study_Hours"],
                df.loc[~pass_mask, "Attendance"],
                color="#F78166", label="Fail", alpha=0.75,
                s=60, edgecolors="#0D1117", linewidths=0.5)
    ax3.set_title("Study Hours vs Attendance")
    ax3.set_xlabel("Study Hours / Day")
    ax3.set_ylabel("Attendance (%)")
    ax3.legend()
    ax3.grid(alpha=0.4)

    # 4. Internal Marks Distribution by Result
    ax4 = fig.add_subplot(gs[1, 0])
    for result, color, label in [("Pass", "#3FB950", "Pass"),
                                  ("Fail", "#F78166", "Fail")]:
        data = df[df["Final_Result"] == result]["Internal_Marks"]
        ax4.hist(data, bins=12, color=color, alpha=0.7,
                 label=label, edgecolor="#0D1117", linewidth=0.6)
    ax4.set_title("Internal Marks by Result")
    ax4.set_xlabel("Internal Marks")
    ax4.set_ylabel("Count")
    ax4.legend()
    ax4.grid(alpha=0.4)

    # 5. Correlation Heatmap
    ax5 = fig.add_subplot(gs[1, 1:])
    num_cols  = ["Attendance", "Internal_Marks", "Assignment_Submission",
                 "Previous_CGPA", "Study_Hours"] + SUBJECTS
    corr_data = df[num_cols].copy()
    corr_data.columns = [c[:8] for c in corr_data.columns]
    corr = corr_data.corr()
    mask = np.triu(np.ones_like(corr, dtype=bool))
    sns.heatmap(corr, mask=mask, cmap="coolwarm", annot=True, fmt=".2f",
                linewidths=0.4, linecolor="#0D1117",
                ax=ax5, annot_kws={"size": 7},
                vmin=-1, vmax=1, cbar_kws={"shrink": 0.8})
    ax5.set_title("Feature Correlation Matrix")
    ax5.tick_params(labelsize=8)

    plt.savefig(f"{OUT_DIR}/08_eda_overview.png")
    plt.show()
    # plt.close()
    print(f"  ✔  EDA overview plot saved → {OUT_DIR}/08_eda_overview.png")

In [ ]:
#  SECTION 8 ▶ PERSONALIZED SUGGESTION ENGINE (Rule-Based Expert System)
def generate_suggestions(student_row: pd.Series, weak_subjects: list) -> dict:
    """
    Rule-Based Expert System:
    Applies a priority-ordered IF-THEN rule set to generate
    personalised academic improvement suggestions.
    """
    suggestions = []
    priority    = "Low"

    # Rule 1: Attendance Critical
    if student_row["Attendance"] < 40:
        suggestions.append("🔴 CRITICAL: Attendance below 40% – risk of being barred from exams. Attend ALL classes immediately.")
        priority = "Critical"
    elif student_row["Attendance"] < 60:
        suggestions.append("🟠 Attendance is low (<60%). Aim for ≥75% to avoid debarment.")
        priority = _higher_priority(priority, "High")
    elif student_row["Attendance"] < 75:
        suggestions.append("🟡 Attendance approaching minimum. Target ≥85% for academic safety.")

    # Rule 2: Assignments
    if student_row["Assignment_Submission"] < 50:
        suggestions.append("🔴 Less than 50% assignments submitted. Complete pending work immediately.")
        priority = _higher_priority(priority, "High")
    elif student_row["Assignment_Submission"] < 75:
        suggestions.append("🟡 Improve assignment completion rate to ≥85% for better internal marks.")

    # Rule 3: Internal Marks
    if student_row["Internal_Marks"] < 40:
        suggestions.append("🔴 Internal marks very low. Consult faculty and revise fundamentals.")
        priority = _higher_priority(priority, "High")
    elif student_row["Internal_Marks"] < 60:
        suggestions.append("🟠 Internal marks need improvement. Practice chapter-end problems regularly.")

    # Rule 4: Weak Subjects
    student_subjects = {s: student_row[s] for s in SUBJECTS}
    personal_weak    = [s for s, sc in student_subjects.items() if sc < WEAK_THRESHOLD]
    if personal_weak:
        suggestions.append(
            f"📚 Weak subjects identified: {', '.join(personal_weak)}. "
            f"Dedicate extra 1-hour sessions to these topics daily."
        )
        priority = _higher_priority(priority, "Medium")

    # Rule 5: Study Hours
    if student_row["Study_Hours"] < 2:
        suggestions.append("⏱ Study time very low (<2h/day). Increase to at least 4-5 hours.")
    elif student_row["Study_Hours"] < 4:
        suggestions.append("⏱ Aim for 5-6 hours of focused study per day.")

    # Rule 6: CGPA Trend
    if student_row["Previous_CGPA"] < 5.0:
        suggestions.append("📊 Previous CGPA is below 5.0. Consider forming a study group.")
    elif student_row["Previous_CGPA"] >= 8.0:
        suggestions.append("⭐ Strong academic record. Explore research/internship opportunities.")

    # Rule 7: Extracurricular balance
    if student_row.get("Extracurricular", "No") == "Yes":
        if student_row["Internal_Marks"] < 60:
            suggestions.append("⚖ Balancing extracurriculars with academics is important. Re-prioritise study time.")

    if not suggestions:
        suggestions.append("✅ Performance is on track. Maintain consistency and target distinction!")
        priority = "Excellent"

    return {"suggestions": suggestions, "priority": priority}

def run_suggestion_demo(df: pd.DataFrame, weak_subjects: list, n_samples: int = 5):
    """Demonstrate the expert system on a sample of students."""
    print("\n" + "═"*62)
    print("  MODULE 8 ▷  PERSONALIZED SUGGESTION ENGINE")
    print("             (Rule-Based Expert System)")
    print("═"*62)

    sample = df.sample(n=n_samples, random_state=7)
    for _, row in sample.iterrows():
        result   = generate_suggestions(row, weak_subjects)
        priority = result["priority"]
        badge    = {"Critical": "🔴", "High": "🟠", "Medium": "🟡",
                    "Low": "🟢", "Excellent": "⭐"}.get(priority, "○")
        print(f"\n  ┌─ Student: {row['Name']:<22} Result: {row['Final_Result']:<6} Priority: {badge} {priority}")
        for sug in result["suggestions"]:
            print(f"  │  {sug}")
        print(f"  └{'─'*58}")

In [ ]:
#  SECTION 9 ▶ PERFORMANCE TREND ANALYSIS 
def performance_trend_analysis(df: pd.DataFrame):
    """
    BONUS FEATURE: Trend Analysis
    Computes composite performance score, categorises students,
    and plots a performance distribution dashboard.
    """
    print("\n" + "═"*62)
    print("  MODULE 9 ▷  PERFORMANCE TREND ANALYSIS  (Bonus AI Feature)")
    print("═"*62)

    df = df.copy()
    # Composite Score = weighted sum of key metrics
    df["Composite_Score"] = (
        df["Attendance"]            * 0.20 +
        df["Internal_Marks"]        * 0.25 +
        df["Assignment_Submission"] * 0.15 +
        df[SUBJECTS].mean(axis=1)   * 0.30 +
        df["Study_Hours"]           * (100/8) * 0.10
    )
    df["Performance_Band"] = pd.cut(
        df["Composite_Score"],
        bins=[0, 40, 55, 70, 85, 100],
        labels=["Critical", "Below Average", "Average", "Good", "Excellent"]
    )
    band_counts = df["Performance_Band"].value_counts().sort_index()
    print(f"\n  Performance Band Distribution:")
    for band, cnt in band_counts.items():
        bar = "█" * cnt
        print(f"  {str(band):<16} {cnt:3d}  {bar}")

    fig, axes = plt.subplots(1, 3, figsize=(18, 6))
    fig.suptitle("Performance Trend & Composite Score Analysis",
                 fontsize=16, color="#F0F6FC", fontweight="bold")

    # Band bar chart
    band_colors = ["#F78166", "#FFA657", "#D2A8FF", "#58A6FF", "#3FB950"]
    ax = axes[0]
    bars = ax.bar(band_counts.index.astype(str), band_counts.values,
                  color=band_colors[:len(band_counts)], alpha=0.85,
                  edgecolor="#0D1117", linewidth=1)
    for bar, val in zip(bars, band_counts.values):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.2,
                str(val), ha="center", fontsize=11, color="#F0F6FC",
                fontweight="bold")
    ax.set_title("Students per Performance Band")
    ax.set_xlabel("Performance Band")
    ax.set_ylabel("Number of Students")
    ax.set_xticklabels(ax.get_xticklabels(), rotation=20, ha="right")
    ax.grid(axis="y", alpha=0.4)

    # Composite Score KDE
    ax2 = axes[1]
    for result, col, lbl in [("Pass", "#3FB950", "Pass"),
                               ("Fail", "#F78166", "Fail")]:
        subset = df[df["Final_Result"] == result]["Composite_Score"]
        if len(subset) > 1:
            subset.plot.kde(ax=ax2, color=col, linewidth=2.5, label=lbl)
    ax2.set_title("Composite Score Distribution (KDE)")
    ax2.set_xlabel("Composite Score")
    ax2.set_ylabel("Density")
    ax2.legend()
    ax2.grid(alpha=0.4)

    # Subject Spider-like bar for top vs bottom students
    ax3 = axes[2]
    top_students = df.nlargest(10, "Composite_Score")[SUBJECTS].mean()
    bot_students = df.nsmallest(10, "Composite_Score")[SUBJECTS].mean()
    x   = np.arange(len(SUBJECTS))
    w   = 0.35
    ax3.bar(x - w/2, top_students, w, label="Top 10 Students",
            color="#3FB950", alpha=0.85, edgecolor="#0D1117")
    ax3.bar(x + w/2, bot_students, w, label="Bottom 10 Students",
            color="#F78166", alpha=0.85, edgecolor="#0D1117")
    ax3.set_xticks(x)
    ax3.set_xticklabels(SUBJECTS, rotation=15, ha="right")
    ax3.set_title("Subject Scores: Top 10 vs Bottom 10")
    ax3.set_ylabel("Average Score")
    ax3.legend()
    ax3.grid(axis="y", alpha=0.4)

    plt.tight_layout()
    plt.savefig(f"{OUT_DIR}/09_trend_analysis.png")
    plt.show()
    # plt.close()
    print(f"  ✔  Trend analysis plot saved → {OUT_DIR}/09_trend_analysis.png")

    return df

In [ ]:
#  SECTION 10 ▶ SUMMARY DASHBOARD
def print_summary(results: dict, subject_means: pd.Series, weak_subjects: list):
    """Print a final structured summary of all results."""
    best_model = max(results, key=lambda k: results[k]["cv_mean"])
    print("\n" + "═"*62)
    print("  FINAL SUMMARY DASHBOARD")
    print("═"*62)
    print(f"\n  📌 DATASET          : 300 students, 14 features")
    print(f"  📌 ALGORITHMS USED  : K-Means, LR, DT, RF, SVM")
    print(f"  📌 BEST CLASSIFIER  : {best_model} (CV mean: {results[best_model]['cv_mean']*100:.1f}%)")
    print(f"\n  MODEL ACCURACY SUMMARY:")
    print(f"    {'Model':<22} {'Test Acc':>10}  {'CV Mean':>10}  {'CV Std':>8}")
    print(f"    {'-'*54}")
    for name, res in results.items():
        star = " ◄ BEST" if name == best_model else ""
        print(f"    {name:<22} {res['accuracy']*100:>9.2f}%  {res['cv_mean']*100:>9.2f}%  {res['cv_std']*100:>7.2f}%{star}")
    print(f"\n  SUBJECT PERFORMANCE:")
    for subj, mean in subject_means.sort_values().items():
        flag = " ⚠ WEAK" if mean < WEAK_THRESHOLD else ""
        print(f"    {subj:<14} {mean:.1f}{flag}")
    print(f"\n  WEAK SUBJECTS       : {weak_subjects or 'None'}")
    print(f"\n  OUTPUT FILES (in ./{OUT_DIR}/):")
    for i in range(1, 10):
        f = f"{OUT_DIR}/0{i}_*.png"
        files = sorted(glob.glob(f))
        for fl in files:
            print(f"    📊  {fl}")
    print("\n" + "═"*62)
    print("  ✅  SMART ACADEMIC MENTOR – EXECUTION COMPLETE")
    print("═"*62 + "\n")


In [ ]:
#  MAIN PIPELINE
# ─────────────────────────────────────────────────────────────────────────────
def main():
    print("\n" + "╔" + "═"*60 + "╗")
    print("║" + "   SMART ACADEMIC MENTOR".center(60) + "║")
    print("║" + "   AI/ML Student Performance Analysis System".center(60) + "║")
    print("║" + "   B.Tech 2nd Year Project  |  CSE (AI/ML)".center(60) + "║")
    print("╚" + "═"*60 + "╝")

    DATA_PATH = "student_data.csv"

    # ── Load ──────────────────────────────────────────────────────────────────
    df_raw = load_data(DATA_PATH)

    # ── EDA ───────────────────────────────────────────────────────────────────
    plot_eda(df_raw)

    # ── Preprocess ────────────────────────────────────────────────────────────
    X_scaled, y, X_df, scaler, le_result, feature_names = preprocess_data(df_raw)

    # ── Unsupervised: Clustering ───────────────────────────────────────────────
    scaler_kmeans = StandardScaler()
    X_kmeans = scaler_kmeans.fit_transform(X_scaled)
    df_for_cluster = df_raw.copy()
    cluster_labels, cluster_map, df_clustered = kmeans_clustering(X_kmeans, df_raw, feature_names)

    # ── Supervised: Classification ────────────────────────────────────────────
    results, X_train, X_test, y_train, y_test = train_and_compare_models(X_scaled, y)

    # ── Feature Importance ───────────────────────────────────────────────────
    plot_feature_importance(results, feature_names)

    # ── Weak Subjects ────────────────────────────────────────────────────────
    weak_subjects, subject_means = identify_weak_subjects(df_raw)

    # ── Personalised Suggestions (Expert System) ──────────────────────────────
    run_suggestion_demo(df_raw, weak_subjects)

    # ── Trend Analysis (Bonus Feature) ───────────────────────────────────────
    df_trend = performance_trend_analysis(df_raw)

    # ── Summary ──────────────────────────────────────────────────────────────
    print_summary(results, subject_means, weak_subjects)


if __name__ == "__main__":
    main()
    plt.show()

In [ ]:
#  SECTION 11 ▶  INTERACTIVE STUDENT PREDICTOR
#  ─────────────────────────────────────────────────────────────────────────────
#  The SVM model was trained on ONLY these 5 features (after preprocessing):
#    → Attendance, Assignment_Submission, Previous_CGPA, Study_Hours, Extracurricular
#
#  Subject marks & Internal_Marks were DROPPED during preprocessing (data leakage).
#  They are NOT needed for Pass/Fail prediction — only for the suggestion engine.
#
#  ✏️  Fill in what you know. Leave subject marks as None if unknown.
#  ─────────────────────────────────────────────────────────────────────────────

def predict_student(
    # ── Identity ──────────────────────────────────────────────────────────────
    name                  = "Test Student",

    # ── PREDICTION FEATURES (required — used by SVM) ──────────────────────────
    attendance            = 72.0,   # % attendance        (0–100)
    assignment_submission = 80.0,   # % assignments done  (0–100)
    previous_cgpa         = 6.8,    # CGPA                (0–10)
    study_hours           = 4.5,    # study hours/day
    extracurricular       = "Yes",  # "Yes" or "No"

    # ── OPTIONAL — only used for personalised suggestions ─────────────────────
    # Leave as None if marks are not yet available
    internal_marks        = None,   # internal assessment (0–100) or None
    maths                 = None,   # subject marks       (0–100) or None
    physics               = None,
    chemistry             = None,
    programming           = None,
    english               = None,
):
    """
    Predicts Pass/Fail using SVM trained on the 5 actual model features.
    Subject marks are completely optional — they only enrich suggestions.
    """

    import warnings, os
    warnings.filterwarnings("ignore")
    import pandas as pd
    import numpy as np
    import matplotlib.pyplot as plt
    import matplotlib.gridspec as gridspec
    from sklearn.preprocessing import StandardScaler, LabelEncoder
    from sklearn.svm import SVC
    from sklearn.cluster import KMeans
    from sklearn.decomposition import PCA

    DATA_PATH     = "student_data.csv"
    SUBJECTS_L    = ["Maths", "Physics", "Chemistry", "Programming", "English"]
    WEAK_THRESH   = 55
    OUT_DIR_L     = "outputs"
    os.makedirs(OUT_DIR_L, exist_ok=True)

    df_raw = pd.read_csv(DATA_PATH)

    # ── Reproduce preprocessing on training data ───────────────────────────────
    df = df_raw.copy()
    df.drop(columns=["StudentID", "Name", "Internal_Marks"] + SUBJECTS_L, inplace=True)
    for col in df.select_dtypes(include=[np.number]).columns:
        df[col].fillna(df[col].mean(), inplace=True)

    le_extra  = LabelEncoder()
    df["Extracurricular"] = le_extra.fit_transform(df["Extracurricular"])
    le_result = LabelEncoder()
    df["Final_Result"] = le_result.fit_transform(df["Final_Result"])

    y  = df["Final_Result"].values
    Xd = df.drop(columns=["Final_Result"])

    corr  = Xd.corr().abs()
    upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
    to_drop = [c for c in upper.columns if any(upper[c] > 0.90)]
    Xd = Xd.drop(columns=to_drop)
    feature_names = list(Xd.columns)

    scaler   = StandardScaler()
    X_scaled = scaler.fit_transform(Xd.values)

    # ── Build the student vector from ONLY the model features ──────────────────
    ext_enc    = 1 if str(extracurricular).strip().lower() == "yes" else 0
    raw_dict   = {
        "Attendance"           : attendance,
        "Assignment_Submission": assignment_submission,
        "Previous_CGPA"        : previous_cgpa,
        "Study_Hours"          : study_hours,
        "Extracurricular"      : ext_enc,
    }
    for f in to_drop:
        raw_dict.pop(f, None)

    student_vec    = np.array([raw_dict.get(f, 0.0) for f in feature_names]).reshape(1, -1)
    student_scaled = scaler.transform(student_vec)

    # ── Train SVM & predict ────────────────────────────────────────────────────
    svm = SVC(kernel="rbf", C=0.5, gamma="scale",
              probability=True, class_weight="balanced", random_state=42)
    svm.fit(X_scaled, y)

    pred_enc   = svm.predict(student_scaled)[0]
    pred_proba = svm.predict_proba(student_scaled)[0]
    pred_label = le_result.inverse_transform([pred_enc])[0]
    confidence = pred_proba[pred_enc] * 100

    # ── KMeans cluster ─────────────────────────────────────────────────────────
    kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
    kmeans.fit(X_scaled)
    cluster_labels_all = kmeans.labels_

    df_temp = df_raw.copy()
    df_temp["Cluster"] = cluster_labels_all
    sorted_cl   = df_temp.groupby("Cluster")["Attendance"].mean().sort_values().index.tolist()
    cluster_map = {sorted_cl[0]: "At-Risk", sorted_cl[1]: "Average", sorted_cl[2]: "High Performer"}

    student_cluster  = kmeans.predict(student_scaled)[0]
    cluster_category = cluster_map[student_cluster]

    pca_model  = PCA(n_components=2)
    X_2d       = pca_model.fit_transform(X_scaled)
    student_2d = pca_model.transform(student_scaled)

    # ── Suggestion engine (marks used only if provided) ────────────────────────
    PRIORITY_RANK_L = {"Low": 0, "Medium": 1, "High": 2, "Critical": 3, "Excellent": -1}
    def _higher(a, b):
        return a if PRIORITY_RANK_L.get(a, 0) >= PRIORITY_RANK_L.get(b, 0) else b

    sugg     = []
    priority = "Low"

    if attendance < 40:
        sugg.append("🔴 CRITICAL: Attendance below 40% – risk of being barred from exams!")
        priority = "Critical"
    elif attendance < 60:
        sugg.append("🟠 Attendance is low (<60%). Aim for ≥75% immediately.")
        priority = _higher(priority, "High")
    elif attendance < 75:
        sugg.append("🟡 Attendance near minimum. Target ≥85%.")

    if assignment_submission < 50:
        sugg.append("🔴 Less than 50% assignments submitted. Catch up immediately.")
        priority = _higher(priority, "High")
    elif assignment_submission < 75:
        sugg.append("🟡 Improve assignment submission rate to ≥85%.")

    if previous_cgpa < 5.0:
        sugg.append("📊 Previous CGPA below 5.0. Consider forming a study group.")
    elif previous_cgpa >= 8.0:
        sugg.append("⭐ Strong academic record! Explore research/internship opportunities.")

    if study_hours < 2:
        sugg.append("⏱ Study time very low (<2h/day). Increase to at least 4–5 hours.")
    elif study_hours < 4:
        sugg.append("⏱ Aim for 5–6 hours of focused study per day.")

    # Marks-based suggestions — only if provided
    subj_scores = {
        "Maths": maths, "Physics": physics, "Chemistry": chemistry,
        "Programming": programming, "English": english
    }
    known_subjects = {s: v for s, v in subj_scores.items() if v is not None}

    if internal_marks is not None:
        if internal_marks < 40:
            sugg.append("🔴 Internal marks very low. Consult faculty and revise fundamentals.")
            priority = _higher(priority, "High")
        elif internal_marks < 60:
            sugg.append("🟠 Internal marks need improvement. Practice problems regularly.")

    if known_subjects:
        personal_weak = [s for s, sc in known_subjects.items() if sc < WEAK_THRESH]
        if personal_weak:
            sugg.append(f"📚 Weak subjects: {', '.join(personal_weak)}. Dedicate 1 extra hour/day to each.")
            priority = _higher(priority, "Medium")
    else:
        personal_weak = []
        sugg.append("📝 Subject marks not provided – suggestion engine using behavioural features only.")

    if str(extracurricular).strip().lower() == "yes":
        if (internal_marks is not None and internal_marks < 60) or attendance < 75:
            sugg.append("⚖ Re-prioritise study over extracurriculars to improve metrics.")

    if not sugg:
        sugg.append("✅ All metrics look good. Maintain consistency and aim for distinction!")
        priority = "Excellent"

    # ── Print report ───────────────────────────────────────────────────────────
    result_icon  = "✅" if pred_label == "Pass" else "❌"
    cluster_icon = {"At-Risk": "⚠️ ", "Average": "📈", "High Performer": "🏆"}.get(cluster_category, "")
    badge_map    = {"Critical": "🔴", "High": "🟠", "Medium": "🟡", "Low": "🟢", "Excellent": "⭐"}

    print("\n" + "╔" + "═"*62 + "╗")
    print("║" + "   🎓  SMART ACADEMIC MENTOR – PREDICTION REPORT".center(62) + "║")
    print("╠" + "═"*62 + "╣")
    print(f"║  Student         : {name:<42}║")
    print(f"║  SVM Prediction  : {result_icon} {pred_label:<40}║")
    print(f"║  Confidence      : {confidence:.1f}%{'':<39}║")
    print(f"║  Cluster Group   : {cluster_icon} {cluster_category:<39}║")
    print("╠" + "═"*62 + "╣")
    print(f"║  {'── PREDICTION FEATURES (used by SVM)':<60}║")
    print(f"║  {'─'*58}  ║")
    feat_display = {
        "Attendance (%)":          (attendance,            df_raw["Attendance"].mean()),
        "Assignment Submission (%)": (assignment_submission, df_raw["Assignment_Submission"].mean()),
        "Previous CGPA":           (previous_cgpa,         df_raw["Previous_CGPA"].mean()),
        "Study Hours / Day":       (study_hours,           df_raw["Study_Hours"].mean()),
    }
    for feat, (val, avg) in feat_display.items():
        arrow = "▲" if val >= avg else "▼"
        print(f"║  {feat:<30} {val:>10.1f}  {avg:>10.1f}{arrow}   ║")
    print(f"║  {'Extracurricular':<30} {str(extracurricular):>22}   ║")

    if known_subjects or internal_marks is not None:
        print("╠" + "═"*62 + "╣")
        print(f"║  {'── OPTIONAL MARKS (for suggestions only)':<60}║")
        print(f"║  {'─'*58}  ║")
        if internal_marks is not None:
            print(f"║  {'Internal Marks':<30} {internal_marks:>10.1f}  {df_raw['Internal_Marks'].mean():>10.1f}{'▲' if internal_marks >= df_raw['Internal_Marks'].mean() else '▼'}   ║")
        for subj, score in known_subjects.items():
            status = " ⚠WEAK" if score < WEAK_THRESH else "   ✔OK"
            print(f"║  {subj:<30} {score:>10.1f}  {status:>12}              ║")

    print("╠" + "═"*62 + "╣")
    print(f"║  SUGGESTIONS   (Priority: {badge_map.get(priority,'')} {priority}){'':<20}║")
    print(f"║  {'─'*58}  ║")
    for s in sugg:
        while len(s) > 57:
            print(f"║  {s[:57]}  ║")
            s = "   " + s[57:]
        print(f"║  {s:<58}║")
    print("╚" + "═"*62 + "╝")

    # ── Visual Dashboard ───────────────────────────────────────────────────────
    cluster_colors = {"At-Risk": "#F78166", "Average": "#FFA657", "High Performer": "#3FB950"}
    pred_color     = "#3FB950" if pred_label == "Pass" else "#F78166"

    fig = plt.figure(figsize=(16, 9))
    fig.suptitle(f"🎓  Prediction Dashboard  –  {name}",
                 fontsize=15, color="#F0F6FC", fontweight="bold")
    gs = gridspec.GridSpec(2, 3, figure=fig, hspace=0.5, wspace=0.4)

    # Plot A: SVM probability
    ax_a = fig.add_subplot(gs[0, 0])
    labels_bar = le_result.classes_
    bar_colors = ["#F78166" if l == "Fail" else "#3FB950" for l in labels_bar]
    bars = ax_a.bar(labels_bar, pred_proba * 100, color=bar_colors,
                    edgecolor="#0D1117", linewidth=1.2, alpha=0.9)
    for bar, prob in zip(bars, pred_proba * 100):
        ax_a.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
                  f"{prob:.1f}%", ha="center", fontsize=12,
                  color="#F0F6FC", fontweight="bold")
    ax_a.set_ylim(0, 120)
    ax_a.set_title(f"SVM Prediction: {result_icon} {pred_label}", pad=10)
    ax_a.set_ylabel("Probability (%)")
    ax_a.grid(axis="y", alpha=0.4)

    # Plot B: PCA cluster scatter
    ax_b = fig.add_subplot(gs[0, 1])
    for cl_idx, cat in cluster_map.items():
        mask = cluster_labels_all == cl_idx
        ax_b.scatter(X_2d[mask, 0], X_2d[mask, 1],
                     label=cat, color=cluster_colors[cat],
                     s=55, alpha=0.55, edgecolors="#0D1117", linewidths=0.4)
    ax_b.scatter(student_2d[0, 0], student_2d[0, 1],
                 color=pred_color, s=280, marker="*", zorder=5,
                 edgecolors="#F0F6FC", linewidths=1.2, label=f"{name[:14]} (You)")
    ax_b.set_title(f"PCA Clusters  –  {cluster_icon} {cluster_category}", pad=10)
    ax_b.set_xlabel("PC1"); ax_b.set_ylabel("PC2")
    ax_b.legend(fontsize=8, loc="best"); ax_b.grid(alpha=0.4)

    # Plot C: Prediction feature bars vs class avg
    ax_c = fig.add_subplot(gs[0, 2])
    feat_labels_c = ["Attendance", "Assignments", "CGPA×10", "StudyHrs×10"]
    student_vals  = [attendance, assignment_submission, previous_cgpa*10, study_hours*10]
    class_avgs_c  = [df_raw["Attendance"].mean(),
                     df_raw["Assignment_Submission"].mean(),
                     df_raw["Previous_CGPA"].mean()*10,
                     df_raw["Study_Hours"].mean()*10]
    xc = np.arange(len(feat_labels_c))
    wc = 0.35
    ax_c.bar(xc - wc/2, student_vals, wc, label=name[:12],
             color=pred_color, alpha=0.85, edgecolor="#0D1117")
    ax_c.bar(xc + wc/2, class_avgs_c, wc, label="Class Avg",
             color="#58A6FF", alpha=0.75, edgecolor="#0D1117")
    ax_c.set_xticks(xc)
    ax_c.set_xticklabels(feat_labels_c, rotation=15, ha="right", fontsize=8)
    ax_c.set_title("You vs Class Average\n(Prediction Features Only)", pad=8)
    ax_c.set_ylabel("Value")
    ax_c.legend(fontsize=8); ax_c.grid(axis="y", alpha=0.4)

    # Plot D: Subject marks (if provided), else show "not provided"
    ax_d = fig.add_subplot(gs[1, 0])
    if known_subjects:
        s_colors = ["#F78166" if v < WEAK_THRESH else "#58A6FF" for v in known_subjects.values()]
        bars_d = ax_d.bar(known_subjects.keys(), known_subjects.values(),
                          color=s_colors, edgecolor="#0D1117", linewidth=1, alpha=0.9)
        ax_d.axhline(WEAK_THRESH, color="#FFA657", linestyle="--",
                     linewidth=1.8, label=f"Weak Threshold ({WEAK_THRESH})")
        ax_d.axhline(df_raw[list(known_subjects.keys())].mean().mean(),
                     color="#D2A8FF", linestyle=":", linewidth=1.5, label="Class Avg")
        for bar, val in zip(bars_d, known_subjects.values()):
            ax_d.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
                      str(int(val)), ha="center", fontsize=10,
                      color="#F0F6FC", fontweight="bold")
        ax_d.set_ylim(0, 115); ax_d.set_ylabel("Marks")
        ax_d.legend(fontsize=8)
        ax_d.set_xticklabels(ax_d.get_xticklabels(), rotation=15, ha="right")
    else:
        ax_d.text(0.5, 0.5, "Subject marks\nnot provided\n\n(optional — only\nneeded for\nsuggestions)",
                  ha="center", va="center", fontsize=13, color="#8B949E",
                  transform=ax_d.transAxes)
        ax_d.set_xticks([]); ax_d.set_yticks([])
    ax_d.set_title("Subject Scores (Optional)", pad=10)
    ax_d.grid(axis="y", alpha=0.3)

    # Plot E: Metric health (prediction features only)
    ax_e = fig.add_subplot(gs[1, 1])
    health_metrics = {
        "Attendance":   min(attendance / 100, 1.0),
        "Assignments":  min(assignment_submission / 100, 1.0),
        "CGPA":         min(previous_cgpa / 10, 1.0),
        "Study Hours":  min(study_hours / 6, 1.0),
    }
    h_colors = ["#F78166" if v < 0.6 else "#3FB950" for v in health_metrics.values()]
    bars_e = ax_e.bar(health_metrics.keys(), [v*100 for v in health_metrics.values()],
                      color=h_colors, edgecolor="#0D1117", linewidth=1, alpha=0.9)
    ax_e.axhline(60, color="#FFA657", linestyle="--", linewidth=1.5, label="60% benchmark")
    for bar, val in zip(bars_e, health_metrics.values()):
        ax_e.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
                  f"{val*100:.0f}%", ha="center", fontsize=10,
                  color="#F0F6FC", fontweight="bold")
    ax_e.set_ylim(0, 115)
    ax_e.set_title("Prediction Feature Health\n(inputs to SVM)", pad=8)
    ax_e.set_ylabel("%"); ax_e.legend(fontsize=8); ax_e.grid(axis="y", alpha=0.4)
    ax_e.set_xticklabels(ax_e.get_xticklabels(), rotation=15, ha="right")

    # Plot F: Confidence gauge
    ax_f = fig.add_subplot(gs[1, 2])
    fail_p = pred_proba[list(le_result.classes_).index("Fail")] * 100 if "Fail" in le_result.classes_ else pred_proba[0]*100
    pass_p = pred_proba[list(le_result.classes_).index("Pass")] * 100 if "Pass" in le_result.classes_ else pred_proba[1]*100
    wedge_colors = ["#F78166", "#3FB950"]
    wedges, texts, autotexts = ax_f.pie(
        [fail_p, pass_p], labels=["Fail", "Pass"],
        colors=wedge_colors, autopct="%1.1f%%", startangle=90,
        wedgeprops={"edgecolor": "#0D1117", "linewidth": 2},
        textprops={"color": "#C9D1D9", "fontsize": 11}
    )
    for at in autotexts:
        at.set_color("#0D1117"); at.set_fontweight("bold"); at.set_fontsize(11)
    ax_f.set_title(f"Confidence Breakdown\n{result_icon} {pred_label} ({confidence:.1f}%)", pad=10)

    plt.savefig(f"{OUT_DIR_L}/11_student_prediction.png", dpi=150)
    plt.show()
    print(f"\n  ✔  Dashboard saved → {OUT_DIR_L}/11_student_prediction.png")

    return {
        "prediction" : pred_label,
        "confidence" : confidence,
        "cluster"    : cluster_category,
        "suggestions": sugg,
        "priority"   : priority,
    }


# ══════════════════════════════════════════════════════════════════════════════
#  ✏️  EDIT VALUES BELOW — only the top 5 fields are needed for prediction
#     Subject marks are OPTIONAL (leave as None if not yet available)
# ══════════════════════════════════════════════════════════════════════════════

result = predict_student(
    # ── Required for Pass/Fail prediction ─────────────────────────────────────
    name                  = "Aryan Sharma",
    attendance            = 72.0,
    assignment_submission = 80.0,
    previous_cgpa         = 6.8,
    study_hours           = 4.5,
    extracurricular       = "Yes",

    # ── Optional: provide if known, else leave as None ─────────────────────────
    internal_marks        = None,   # e.g. 58.0  or  None
    maths                 = None,   # e.g. 62.0  or  None
    physics               = None,
    chemistry             = None,
    programming           = None,
    english               = None,
)
